In [20]:
import os
import numpy as np
import pandas as pd

In [21]:
# Import data:
# run script from the data_input file
os.chdir('C:/Users/p528552/OneDrive - South African Reserve Bank/Documents/1. MEng - Data Science/1. Project_2025/Data/factor_timing/data_input')


In [22]:
cluster_reps = pd.read_csv("representatives.csv")

features = pd.read_csv("featureset.csv", parse_dates=['Date'],na_values=['NA'])
features = features.set_index('Date')

add_features = pd.read_csv("additional_features.csv", parse_dates=['Date'],na_values=['NA'])
add_features = add_features.set_index('Date')




In [23]:
# Ensure column 2 of dataset_a is named properly; let's call it 'col2':
col2_values = cluster_reps['Representative'].unique()

# Filter dataset_b by checking whether its index exists in col2_values:
data= features[features.columns.intersection(col2_values)]

print(data)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-03-05        -33.663206          8.061237            91.3   
2007-03-12        -26.590916          9.236760            91.3   
2007-03-19        -27.816286          9.407570            91.3   
2007-03-26        -25.510835          9.096880            91.3   
2007-04-02        -27.546502          9.140546            88.4   
...                      ...               ...             ...   
2025-03-03         44.421245         -5.261720            64.7   
2025-03-10         51.117954         -3.610349            64.7   
2025-03-17         42.305038         -6.657027            64.7   
2025-03-24         38.142130         -7.595933            64.7   
2025-03-31         44.285942         -6.064842            57.0   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [24]:
# --- Drop features with more than 30% missing values ---
threshold = 0.10
missing_ratio = data.isnull().mean()
cols_to_drop = missing_ratio[missing_ratio > threshold].index
data = data.drop(columns=cols_to_drop)

print(data)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2007-03-05        -33.663206          8.061237            91.3   
2007-03-12        -26.590916          9.236760            91.3   
2007-03-19        -27.816286          9.407570            91.3   
2007-03-26        -25.510835          9.096880            91.3   
2007-04-02        -27.546502          9.140546            88.4   
...                      ...               ...             ...   
2025-03-03         44.421245         -5.261720            64.7   
2025-03-10         51.117954         -3.610349            64.7   
2025-03-17         42.305038         -6.657027            64.7   
2025-03-24         38.142130         -7.595933            64.7   
2025-03-31         44.285942         -6.064842            57.0   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [25]:
df = pd.concat([data,add_features], axis=1, join='inner')
lagged_data = df.shift(4).reset_index()
lagged_data = lagged_data.set_index('Date')
lagged_data= lagged_data.dropna()

print(lagged_data)


            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2008-06-30         43.764525          6.157399            59.8   
2008-07-07         47.865713          4.654912            59.8   
2008-07-14         44.722107          5.007951            59.8   
2008-07-21         46.645075          4.146473            59.8   
2008-07-28         47.943107          3.404113            56.4   
...                      ...               ...             ...   
2024-01-01         32.753330         -9.716542            61.3   
2024-01-08         31.765017         -9.542882            61.3   
2024-01-15         31.849078         -9.517372            61.3   
2024-01-22         32.957527         -9.821722            61.3   
2024-01-29         31.661633         -9.617528            69.7   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [26]:
# View cluster representatives
lagged_data.to_csv('lagged_data.csv',index=True)

In [27]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# df_features = your feature dataset (DataFrame)
# df_target   = your target dataset (DataFrame or Series)

scaler = StandardScaler()

# Fit on features only
X_scaled = scaler.fit_transform(lagged_data)

# Convert back to DataFrame with the same columns + index
X_scaled = pd.DataFrame(X_scaled, 
                        columns=lagged_data.columns, 
                        index=lagged_data.index)


print(X_scaled)

            VolTermStructPC1  VolTermStructPC2  MichgnConcIndx  \
Date                                                             
2008-06-30         -0.027473          2.365160       -1.449051   
2008-07-07          0.238040          2.088292       -1.449051   
2008-07-14          0.034522          2.153348       -1.449051   
2008-07-21          0.159016          1.994601       -1.449051   
2008-07-28          0.243051          1.857804       -1.699551   
...                      ...               ...             ...   
2024-01-01         -0.740343         -0.559978       -1.338535   
2024-01-08         -0.804327         -0.527977       -1.338535   
2024-01-15         -0.798885         -0.523276       -1.338535   
2024-01-22         -0.727123         -0.579360       -1.338535   
2024-01-29         -0.811020         -0.541732       -0.719651   

            RandPPP_Factor_Inst_TS_ST  BCMPEBLS.Index  PPP.ZA.Index_QS  \
Date                                                               

In [29]:
# View cluster representatives
X_scaled.to_csv('scaled_data.csv',index=True)